In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas11_PySpark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

print(f"Total baris dataset: {df.count()}")
df.printSchema()
df.show(5)

Total baris dataset: 2500
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



**A. Feature Engineering**

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx", handleInvalid="keep")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="keep")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx", handleInvalid="keep")

df = indexer_menu.fit(df).transform(df)
df = indexer_cabang.fit(df).transform(df)
df = indexer_waktu.fit(df).transform(df)

assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df = assembler_utama.transform(df)
print("Berhasil melakukan Feature Engineering (Bagian A)!")
df.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "features").show(5, truncate=False)

Berhasil melakukan Feature Engineering (Bagian A)!
+--------------+----------+-----------+---+------------+-------------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|features                 |
+--------------+----------+-----------+---+------------+-------------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |[1.0,30000.0,5.0,2.0,1.0]|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |[3.0,20000.0,5.0,1.0,3.0]|
|Kentang Goreng|Semarang  |Siang      |5  |22000       |[5.0,22000.0,5.0,2.0,2.0]|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |[2.0,20000.0,5.0,1.0,0.0]|
|Roti Bakar    |Magelang  |Malam      |2  |30000       |[2.0,30000.0,4.0,0.0,1.0]|
+--------------+----------+-----------+---+------------+-------------------------+
only showing top 5 rows



**B. Train/Test Split & Pelatihan Model**

In [4]:
from pyspark.ml.regression import LinearRegression

train_df, test_df = df.randomSplit([0.75, 0.25], seed=111)

print(f"Jumlah baris data training: {train_df.count()}")
print(f"Jumlah baris data testing: {test_df.count()}")

lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")
lr_utama_model = lr_utama.fit(train_df)

print("Model LinearRegression utama berhasil dilatih!")

Jumlah baris data training: 1888


Jumlah baris data testing: 612


[Stage 29:>                                                         (0 + 1) / 1]

Model LinearRegression utama berhasil dilatih!


**C. Evaluasi Model**

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions_utama = lr_utama_model.transform(test_df)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = evaluator_rmse.evaluate(predictions_utama)
r2_utama = evaluator_r2.evaluate(predictions_utama)

print(f"Evaluasi Model Utama (5 Fitur):")
print(f" - RMSE : {rmse_utama}")
print(f" - R²   : {r2_utama}")

Evaluasi Model Utama (5 Fitur):
 - RMSE : 10225.469264160718
 - R²   : 0.9180834820426679


**D. Eksperimen Perbandingan Fitur**

In [6]:
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)
df_sederhana = assembler_sederhana.transform(df)

train_sederhana, test_sederhana = df_sederhana.randomSplit([0.75, 0.25], seed=111)

lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
lr_sederhana_model = lr_sederhana.fit(train_sederhana)

predictions_sederhana = lr_sederhana_model.transform(test_sederhana)
r2_sederhana = evaluator_r2.evaluate(predictions_sederhana)

print(f"R² Model Utama (5 Fitur)    : {r2_utama}")
print(f"R² Model Kedua (2 Fitur)    : {r2_sederhana}")

R² Model Utama (5 Fitur)    : 0.9180834820426679
R² Model Kedua (2 Fitur)    : 0.917822110497783


**E. Prediksi Skenario Baru**

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType


skenario_data = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000.0),
    ("Cappuccino", "Yogyakarta", "Siang", 1, 25000.0),
    ("Roti Bakar", "Semarang", "Malam", 3, 20000.0)
]

schema = StructType([
    StructField("nama_menu", StringType(), True),
    StructField("cabang", StringType(), True),
    StructField("waktu_pesan", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", DoubleType(), True)
])

df_skenario = spark.createDataFrame(skenario_data, schema)


indexer_menu_skenario = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx_baru", handleInvalid="keep").fit(df)
indexer_cabang_skenario = StringIndexer(inputCol="cabang", outputCol="cabang_idx_baru", handleInvalid="keep").fit(df)
indexer_waktu_skenario = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx_baru", handleInvalid="keep").fit(df)

df_skenario = indexer_menu_skenario.transform(df_skenario)
df_skenario = indexer_cabang_skenario.transform(df_skenario)
df_skenario = indexer_waktu_skenario.transform(df_skenario)

assembler_skenario = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx_baru", "cabang_idx_baru", "waktu_pesan_idx_baru"],
    outputCol="features"
)

df_skenario = assembler_skenario.transform(df_skenario)


hasil_prediksi = lr_utama_model.transform(df_skenario)


print("Hasil Prediksi Skenario Pesanan Baru:")
hasil_prediksi.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction") \
    .withColumnRenamed("prediction", "prediksi_total_penjualan") \
    .show(truncate=False)